# LeetCode 496 — Next Greater Element I bằng Python

**Link đề:** https://leetcode.com/problems/next-greater-element-i/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Stack, Monotonic Stack

```python
class Solution:
    def nextGreaterElement(self, nums1: list[int], nums2: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu đúng khái niệm **phần tử lớn hơn kế tiếp** (next greater element) — "đầu tiên, bên phải, lớn hơn".
2. Ôn `list`, `dict`, `list.index`, cách dùng `list` làm **stack** (`append`, `pop`, `stack[-1]`).
3. Viết lời giải trực giác O(m·n), rồi thấy nút thắt "quét đi quét lại".
4. Học kỹ thuật **monotonic stack** (stack đơn điệu giảm) để giải trong O(n + m).
5. Trace stack từng bước, chứng minh vì sao đúng, so sánh 3 cách, tự luyện biến thể (vòng tròn, Daily Temperatures).

## 1. Đề bài

**Next greater element** của một phần tử `x` trong mảng là phần tử **đầu tiên ở bên phải** `x` mà **lớn hơn** `x`.

Cho hai mảng **không có phần tử trùng** `nums1`, `nums2`, trong đó mọi phần tử của `nums1` đều xuất hiện trong `nums2`.
Với mỗi `nums1[i]`, tìm vị trí `j` sao cho `nums2[j] == nums1[i]`, rồi tìm next greater element của `nums2[j]` **trong `nums2`**. Không có thì trả `-1`.

Trả về mảng `ans` cùng độ dài `nums1`.

| nums1 | nums2 | Output | Giải thích |
|---|---|---|---|
| `[4,1,2]` | `[1,3,4,2]` | `[-1,3,-1]` | 4 không có ai lớn hơn bên phải; 1 → 3; 2 đứng cuối |
| `[2,4]` | `[1,2,3,4]` | `[3,-1]` | 2 → 3; 4 đứng cuối |

**Constraints:**
- `1 <= nums1.length <= nums2.length <= 1000`
- `0 <= nums1[i], nums2[i] <= 10^4`
- Mọi số trong `nums1` và `nums2` là **duy nhất** (unique); mọi số của `nums1` đều có trong `nums2`.

**Follow up:** làm được trong `O(nums1.length + nums2.length)` không?

### Đọc kỹ 3 chữ: "đầu tiên", "bên phải", "lớn hơn"

Với `nums2 = [1, 3, 4, 2]`, xét số `1`:
- Bên phải `1` là `3, 4, 2`.
- Các số lớn hơn `1`: `3, 4, 2` — nhưng ta lấy số **đầu tiên** gặp được: `3`.
- **Không phải** số lớn nhất (`4`), cũng **không phải** số lớn hơn nhỏ nhất (`2`).

Ta viết một cell nhỏ để "nhìn" phần bên phải của từng phần tử.

In [1]:
nums2 = [1, 3, 4, 2]
for j, x in enumerate(nums2):
    right_part = nums2[j + 1:]
    bigger = [y for y in right_part if y > x]
    first = bigger[0] if bigger else -1
    print(f"x={x}: bên phải={right_part}, các số lớn hơn={bigger}, next greater={first}")

x=1: bên phải=[3, 4, 2], các số lớn hơn=[3, 4, 2], next greater=3
x=3: bên phải=[4, 2], các số lớn hơn=[4], next greater=4
x=4: bên phải=[2], các số lớn hơn=[], next greater=-1
x=2: bên phải=[], các số lớn hơn=[], next greater=-1


## 2. Nền tảng Python cần dùng

### 2.1. `enumerate` và slicing `a[j+1:]`

- `enumerate(nums2)` cho từng cặp `(chỉ số, giá trị)`.
- `nums2[j + 1:]` là **slicing**: lấy từ vị trí `j+1` tới hết. Nếu `j` là vị trí cuối thì kết quả là list rỗng `[]`.

In [2]:
a = [10, 20, 30]
for i, v in enumerate(a):
    print(i, v, "-> phần bên phải:", a[i + 1:])

0 10 -> phần bên phải: [20, 30]
1 20 -> phần bên phải: [30]
2 30 -> phần bên phải: []


### 2.2. `list.index(x)` — tìm vị trí của một giá trị

`nums2.index(4)` trả về chỉ số **đầu tiên** có giá trị 4. Nó phải duyệt từ trái sang nên tốn `O(n)`.
Vì đề bảo các số **unique**, vị trí tìm được là duy nhất.

In [3]:
nums2 = [1, 3, 4, 2]
print(nums2.index(4))   # 2
print(nums2.index(1))   # 0

2
0


### 2.3. `dict` — tra cứu giá trị → kết quả trong O(1) trung bình

Ta sẽ lưu `next_greater[x] = y` nghĩa là "next greater của x là y".

- Tạo: `d = {}`
- Gán: `d[x] = y`
- Đọc: `d[x]`; đọc có mặc định: `d.get(x, -1)` (nếu không có key thì trả `-1`).

In [4]:
d = {}
d[1] = 3
d[3] = 4
print(d)              # {1: 3, 3: 4}
print(d[1])           # 3
print(d.get(4, -1))   # 4 chưa có -> -1
print(3 in d, 4 in d) # True False

{1: 3, 3: 4}
3
-1
True False


### 2.4. `list` dùng làm **stack** (ngăn xếp)

Stack giống chồng đĩa: **vào sau, ra trước** (LIFO — Last In First Out).

| Thao tác | Cú pháp | Ý nghĩa |
|---|---|---|
| đẩy vào đỉnh | `stack.append(x)` | O(1) |
| xem đỉnh | `stack[-1]` | O(1), **lỗi nếu stack rỗng** |
| lấy ra khỏi đỉnh | `stack.pop()` | O(1), trả về phần tử đỉnh |
| kiểm tra rỗng | `if stack:` / `while stack:` | list rỗng có giá trị "False" |

In [5]:
stack = []
stack.append(5)
stack.append(2)
stack.append(1)
print("stack:", stack, "| đỉnh:", stack[-1])
top = stack.pop()
print("pop ra:", top, "| còn:", stack)
print("stack rỗng?", not stack)
while stack:
    print("pop", stack.pop())
print("bây giờ rỗng?", not stack)

stack: [5, 2, 1] | đỉnh: 1
pop ra: 1 | còn: [5, 2]
stack rỗng? False
pop 2
pop 5
bây giờ rỗng? True


## 3. Cách 1 — Trực giác (Brute Force)

Làm đúng như đề nói:
1. Với mỗi `x` trong `nums1`, tìm vị trí `j` của `x` trong `nums2`.
2. Từ `j + 1` đi sang phải, gặp số đầu tiên lớn hơn `x` thì dừng.
3. Không gặp → `-1`.

In [6]:
def next_greater_bruteforce(nums1, nums2):
    ans = []
    for x in nums1:
        j = nums2.index(x)          # O(n)
        found = -1
        for k in range(j + 1, len(nums2)):
            if nums2[k] > x:
                found = nums2[k]
                break               # chỉ lấy số ĐẦU TIÊN
        ans.append(found)
    return ans

print(next_greater_bruteforce([4, 1, 2], [1, 3, 4, 2]))  # [-1, 3, -1]
print(next_greater_bruteforce([2, 4], [1, 2, 3, 4]))     # [3, -1]

[-1, 3, -1]
[3, -1]


### Độ phức tạp của Brute Force

Đặt `m = len(nums1)`, `n = len(nums2)`.
- Với mỗi trong `m` phần tử: `index` tốn tới `n`, quét phải tốn tới `n` → `O(m · n)`.
- Bộ nhớ: `O(1)` ngoài mảng kết quả.

Với `n ≤ 1000` cách này **vẫn được chấp nhận** trên LeetCode. Nhưng follow up hỏi `O(m + n)`, và ý tưởng tối ưu là một **pattern rất quan trọng**.

### Cách 1b — Bỏ `index` bằng dict vị trí

Một cải tiến nhỏ: dựng trước `pos[value] = index` để tìm `j` trong O(1). Phần quét phải vẫn O(n) mỗi truy vấn.

In [7]:
def next_greater_pos_map(nums1, nums2):
    pos = {v: i for i, v in enumerate(nums2)}   # dict comprehension
    ans = []
    for x in nums1:
        found = -1
        for k in range(pos[x] + 1, len(nums2)):
            if nums2[k] > x:
                found = nums2[k]
                break
        ans.append(found)
    return ans

print({v: i for i, v in enumerate([1, 3, 4, 2])})
print(next_greater_pos_map([4, 1, 2], [1, 3, 4, 2]))

{1: 0, 3: 1, 4: 2, 2: 3}
[-1, 3, -1]


`{v: i for i, v in enumerate(nums2)}` là **dict comprehension**: viết gọn của vòng lặp "với mỗi `(i, v)`, gán `d[v] = i`".

## 4. Nút thắt: ta đang quét đi quét lại

Xét `nums2 = [5, 4, 3, 2, 1, 6]` và `nums1 = nums2` (hỏi hết).
- Với 5: quét 4, 3, 2, 1, 6 → 6.
- Với 4: quét 3, 2, 1, 6 → 6.
- Với 3: quét 2, 1, 6 → 6. ...

Đoạn `..., 1, 6` bị quét lại nhiều lần. Cả 5 số đều "chờ" cùng một số `6`. Đếm số lần so sánh để thấy rõ:

In [8]:
def count_comparisons_bruteforce(nums2):
    comps = 0
    for j, x in enumerate(nums2):
        for k in range(j + 1, len(nums2)):
            comps += 1
            if nums2[k] > x:
                break
    return comps

for n in [6, 50, 500, 1000]:
    worst = list(range(n - 1, 0, -1)) + [n]   # giảm dần rồi 1 số lớn ở cuối
    print(f"n={n:5d}: số so sánh = {count_comparisons_bruteforce(worst)}")

n=    6: số so sánh = 15
n=   50: số so sánh = 1225
n=  500: số so sánh = 124750
n= 1000: số so sánh = 499500


Số so sánh tăng cỡ `n²/2`. Ta cần một cách mà **mỗi phần tử chỉ được xử lý một vài lần**.

## 5. Insight — Monotonic Stack (stack đơn điệu giảm)

Đổi góc nhìn: thay vì *mỗi số đi tìm người lớn hơn nó*, hãy để **mỗi số mới đến "giải quyết" các số đang chờ**.

Duyệt `nums2` từ trái sang phải, giữ một **stack các số chưa tìm được next greater**:
- Khi gặp số mới `x`: mọi số đang chờ ở đỉnh stack mà **nhỏ hơn `x`** thì `x` chính là next greater của chúng → pop ra, ghi kết quả.
- Sau đó đẩy `x` vào stack (nó bắt đầu chờ).

**Quan sát then chốt:** các số trong stack luôn **giảm dần từ đáy lên đỉnh**. Vì sao? Nếu có một số nhỏ hơn nằm dưới một số lớn hơn, thì khi số lớn hơn được đẩy vào, nó đã pop số nhỏ kia rồi.

Nhờ stack giảm dần, khi `x` đến, ta chỉ cần nhìn **đỉnh**: nếu đỉnh `>= x` thì mọi số bên dưới còn lớn hơn → dừng ngay.

In [9]:
# Thí nghiệm nhỏ: nhìn stack khi đẩy lần lượt 5, 3, 4
stack = []
for x in [5, 3, 4]:
    while stack and stack[-1] < x:
        print(f"  {x} là next greater của {stack.pop()}")
    stack.append(x)
    print(f"sau khi xử lý {x}: stack = {stack}")

sau khi xử lý 5: stack = [5]
sau khi xử lý 3: stack = [5, 3]
  4 là next greater của 3
sau khi xử lý 4: stack = [5, 4]


## 6. Lời giải chính — mổ xẻ từng đoạn

### Đoạn 1: chuẩn bị
```python
next_greater = {}   # value -> next greater value
stack = []          # các value đang chờ, giảm dần từ đáy lên đỉnh
```

### Đoạn 2: duyệt nums2, "giải quyết" các số đang chờ
```python
for x in nums2:
    while stack and stack[-1] < x:
        next_greater[stack.pop()] = x
    stack.append(x)
```
- `while stack and ...`: **phải kiểm tra `stack` trước** vì `stack[-1]` trên list rỗng gây `IndexError`. Toán tử `and` dừng sớm (short-circuit) nên an toàn.
- Dùng `while` chứ không phải `if`: một số `x` có thể giải quyết **nhiều** số đang chờ cùng lúc.

### Đoạn 3: những số còn lại trong stack không có next greater
Ta không cần làm gì: khi trả lời dùng `next_greater.get(x, -1)`.

### Đoạn 4: trả lời truy vấn
```python
return [next_greater.get(x, -1) for x in nums1]
```

In [10]:
def next_greater_stack(nums1, nums2):
    next_greater = {}
    stack = []
    for x in nums2:
        while stack and stack[-1] < x:
            next_greater[stack.pop()] = x
        stack.append(x)
    return [next_greater.get(x, -1) for x in nums1]

print(next_greater_stack([4, 1, 2], [1, 3, 4, 2]))  # [-1, 3, -1]
print(next_greater_stack([2, 4], [1, 2, 3, 4]))     # [3, -1]

[-1, 3, -1]
[3, -1]


## 7. TRACE từng bước

Ví dụ khó hơn để thấy `while` pop nhiều lần: `nums2 = [2, 7, 3, 5, 4, 6, 8]`, `nums1 = [3, 4, 8, 2]`.

| bước | x | stack trước | pop (x là next greater của) | stack sau |
|---:|---:|---|---|---|
| 1 | 2 | `[]` | — | `[2]` |
| 2 | 7 | `[2]` | 2 → 7 | `[7]` |
| 3 | 3 | `[7]` | — (7 ≥ 3) | `[7, 3]` |
| 4 | 5 | `[7, 3]` | 3 → 5 | `[7, 5]` |
| 5 | 4 | `[7, 5]` | — | `[7, 5, 4]` |
| 6 | 6 | `[7, 5, 4]` | 4 → 6, 5 → 6 | `[7, 6]` |
| 7 | 8 | `[7, 6]` | 6 → 8, 7 → 8 | `[8]` |

Cuối cùng `8` còn trong stack → không có next greater → `-1`.
Kết quả cho `nums1 = [3, 4, 8, 2]` là `[5, 6, -1, 7]`.

In [11]:
def trace_next_greater(nums1, nums2):
    next_greater = {}
    stack = []
    for step, x in enumerate(nums2, start=1):
        before = stack.copy()
        popped = []
        while stack and stack[-1] < x:
            v = stack.pop()
            next_greater[v] = x
            popped.append(v)
        stack.append(x)
        note = ", ".join(f"{v}->{x}" for v in popped) or "—"
        print(f"bước {step}: x={x:2d} | stack trước={str(before):12s} | gán: {note:10s} | stack sau={stack}")
    print("Còn chờ (không có next greater):", stack)
    print("next_greater =", next_greater)
    ans = [next_greater.get(x, -1) for x in nums1]
    print("ans =", ans)
    return ans

trace_next_greater([3, 4, 8, 2], [2, 7, 3, 5, 4, 6, 8])

bước 1: x= 2 | stack trước=[]           | gán: —          | stack sau=[2]
bước 2: x= 7 | stack trước=[2]          | gán: 2->7       | stack sau=[7]
bước 3: x= 3 | stack trước=[7]          | gán: —          | stack sau=[7, 3]
bước 4: x= 5 | stack trước=[7, 3]       | gán: 3->5       | stack sau=[7, 5]
bước 5: x= 4 | stack trước=[7, 5]       | gán: —          | stack sau=[7, 5, 4]
bước 6: x= 6 | stack trước=[7, 5, 4]    | gán: 4->6, 5->6 | stack sau=[7, 6]
bước 7: x= 8 | stack trước=[7, 6]       | gán: 6->8, 7->8 | stack sau=[8]
Còn chờ (không có next greater): [8]
next_greater = {2: 7, 3: 5, 4: 6, 5: 6, 6: 8, 7: 8}
ans = [5, 6, -1, 7]


[5, 6, -1, 7]

Trace ví dụ 1 của đề để đối chiếu với bảng mong đợi `[-1, 3, -1]`:

In [12]:
trace_next_greater([4, 1, 2], [1, 3, 4, 2])

bước 1: x= 1 | stack trước=[]           | gán: —          | stack sau=[1]
bước 2: x= 3 | stack trước=[1]          | gán: 1->3       | stack sau=[3]
bước 3: x= 4 | stack trước=[3]          | gán: 3->4       | stack sau=[4]
bước 4: x= 2 | stack trước=[4]          | gán: —          | stack sau=[4, 2]
Còn chờ (không có next greater): [4, 2]
next_greater = {1: 3, 3: 4}
ans = [-1, 3, -1]


[-1, 3, -1]

## 8. Cách 3 — Duyệt từ phải sang trái (stack giữ "ứng viên")

Một cách nhìn khác, cũng O(n): đi từ **phải sang trái**. Stack giữ các số ở bên phải có thể là đáp án.
Khi xét `x`:
1. Pop mọi số `<= x` ra khỏi stack — chúng bị `x` "che khuất", không bao giờ là next greater của ai ở bên trái `x` nữa (vì `x` đứng trước và lớn hơn hoặc bằng).
2. Đỉnh còn lại (nếu có) là next greater của `x`.
3. Đẩy `x` vào.

In [13]:
def next_greater_right_to_left(nums1, nums2):
    next_greater = {}
    stack = []
    for x in reversed(nums2):
        while stack and stack[-1] <= x:
            stack.pop()
        next_greater[x] = stack[-1] if stack else -1
        stack.append(x)
    return [next_greater[x] for x in nums1]

print(next_greater_right_to_left([4, 1, 2], [1, 3, 4, 2]))
print(next_greater_right_to_left([3, 4, 8, 2], [2, 7, 3, 5, 4, 6, 8]))

[-1, 3, -1]
[5, 6, -1, 7]


`reversed(nums2)` cho ra các phần tử theo thứ tự ngược mà **không tạo list mới**. Biểu thức `A if điều_kiện else B` là **toán tử 3 ngôi**.

## 9. Code nộp LeetCode

### Bản Brute Force (đủ qua vì n ≤ 1000)

In [14]:
class Solution:
    def nextGreaterElement(self, nums1: list[int], nums2: list[int]) -> list[int]:
        ans = []
        for x in nums1:
            j = nums2.index(x)
            found = -1
            for k in range(j + 1, len(nums2)):
                if nums2[k] > x:
                    found = nums2[k]
                    break
            ans.append(found)
        return ans

print(Solution().nextGreaterElement([4, 1, 2], [1, 3, 4, 2]))

[-1, 3, -1]


### Bản Monotonic Stack (khuyên dùng — đáp ứng follow up)

In [15]:
class Solution:
    def nextGreaterElement(self, nums1: list[int], nums2: list[int]) -> list[int]:
        next_greater = {}
        stack = []
        for x in nums2:
            while stack and stack[-1] < x:
                next_greater[stack.pop()] = x
            stack.append(x)
        return [next_greater.get(x, -1) for x in nums1]

s = Solution()
print(s.nextGreaterElement([4, 1, 2], [1, 3, 4, 2]))  # [-1, 3, -1]
print(s.nextGreaterElement([2, 4], [1, 2, 3, 4]))     # [3, -1]

[-1, 3, -1]
[3, -1]


## 10. Vì sao thuật toán đúng?

**Invariant (bất biến)** sau khi xử lý xong `nums2[0..j]`:
1. Stack chứa đúng những số trong `nums2[0..j]` **chưa có** next greater trong đoạn đó.
2. Các số trong stack **giảm dần** từ đáy lên đỉnh.

Chứng minh từng ý:
- Khi `x` đến, một số `v` đang chờ mà `v < x`: mọi số nằm giữa `v` và `x` đều **nhỏ hơn `v`** (nếu có số nào lớn hơn `v` thì `v` đã bị pop sớm hơn). Vậy `x` là số **đầu tiên** bên phải `v` lớn hơn `v` → gán đúng.
- Ta pop từ đỉnh và dừng khi đỉnh `>= x`. Vì stack giảm dần, mọi số bên dưới đỉnh còn `>= x` → không ai trong số đó có next greater là `x`. Không bỏ sót.
- Đẩy `x` vào sau khi pop: đỉnh cũ `> x` (số unique) nên stack vẫn giảm dần.
- Hết mảng: số còn trong stack không có ai lớn hơn ở bên phải → `-1`.

## 11. Độ phức tạp

Đặt `n = len(nums2)`, `m = len(nums1)`.
- **Time** của monotonic stack: mỗi phần tử được `append` đúng 1 lần và `pop` tối đa 1 lần → tổng số thao tác stack ≤ `2n`. Dù có `while` lồng trong `for`, tổng vẫn `O(n)` (phân tích **amortized** — khấu hao). Trả lời `m` truy vấn bằng dict: `O(m)` trung bình. Tổng `O(n + m)`.
- **Space**: dict + stack tối đa `n` phần tử → `O(n)`.

| Cách | Time | Space | Ghi chú |
|---|---:|---:|---|
| Brute Force + `index` | `O(m·n)` | `O(1)` | dễ nghĩ nhất |
| Brute Force + dict vị trí | `O(m·n)` | `O(n)` | bỏ được `index`, vẫn quét phải |
| Monotonic stack trái→phải | `O(n + m)` | `O(n)` | pattern chuẩn |
| Monotonic stack phải→trái | `O(n + m)` | `O(n)` | tính trực tiếp cho từng phần tử |

Đếm thao tác stack để kiểm chứng "≤ 2n":

In [16]:
def count_stack_ops(nums2):
    ops = 0
    stack = []
    for x in nums2:
        while stack and stack[-1] < x:
            stack.pop(); ops += 1
        stack.append(x); ops += 1
    return ops

for n in [6, 50, 500, 1000]:
    worst = list(range(n - 1, 0, -1)) + [n]
    print(f"n={n:5d}: thao tác stack = {count_stack_ops(worst)} (<= 2n = {2*n})")

n=    6: thao tác stack = 11 (<= 2n = 12)
n=   50: thao tác stack = 99 (<= 2n = 100)
n=  500: thao tác stack = 999 (<= 2n = 1000)
n= 1000: thao tác stack = 1999 (<= 2n = 2000)


## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Các nhóm cần test:
1. Ví dụ đề.
2. `nums1` chỉ 1 phần tử, `nums2` chỉ 1 phần tử.
3. `nums2` tăng dần (ai cũng có next greater ngay cạnh), giảm dần (không ai có).
4. Phản ví dụ "lấy số lớn nhất bên phải" — next greater phải là số **đầu tiên**: `[1, 5, 3]` với `x = 1` → `5`; với `[1, 3, 5]` → `3` chứ không phải `5`.
5. Đối chiếu 4 cách trên nhiều input ngẫu nhiên (có `seed` để tái lập).

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.nextGreaterElement([4, 1, 2], [1, 3, 4, 2]), [-1, 3, -1])
check("đề 2", sol.nextGreaterElement([2, 4], [1, 2, 3, 4]), [3, -1])
check("một phần tử", sol.nextGreaterElement([7], [7]), [-1])
check("tăng dần", sol.nextGreaterElement([1, 2, 3], [1, 2, 3, 4]), [2, 3, 4])
check("giảm dần", sol.nextGreaterElement([4, 2], [4, 3, 2, 1]), [-1, -1])
check("số đầu tiên, không phải lớn nhất", sol.nextGreaterElement([1], [1, 3, 5]), [3])
check("trace dài", sol.nextGreaterElement([3, 4, 8, 2], [2, 7, 3, 5, 4, 6, 8]), [5, 6, -1, 7])
check("có số 0", sol.nextGreaterElement([0, 10000], [10000, 0, 1]), [1, -1])

PASS đề 1: [-1, 3, -1]
PASS đề 2: [3, -1]
PASS một phần tử: [-1]
PASS tăng dần: [2, 3, 4]
PASS giảm dần: [-1, -1]
PASS số đầu tiên, không phải lớn nhất: [3]
PASS trace dài: [5, 6, -1, 7]
PASS có số 0: [1, -1]


In [18]:
import random
random.seed(496)
for t in range(300):
    n = random.randint(1, 30)
    nums2 = random.sample(range(0, 100), n)          # unique
    m = random.randint(1, n)
    nums1 = random.sample(nums2, m)
    r1 = next_greater_bruteforce(nums1, nums2)
    r2 = next_greater_pos_map(nums1, nums2)
    r3 = next_greater_stack(nums1, nums2)
    r4 = next_greater_right_to_left(nums1, nums2)
    assert r1 == r2 == r3 == r4, (nums1, nums2, r1, r2, r3, r4)
print("300 bộ ngẫu nhiên: 4 cách cho cùng kết quả — PASS")

300 bộ ngẫu nhiên: 4 cách cho cùng kết quả — PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: dùng `if` thay vì `while`
```python
for x in nums2:
    if stack and stack[-1] < x:     # SAI: chỉ pop 1 phần tử
        next_greater[stack.pop()] = x
    stack.append(x)
```
Với `[5, 4, 6]`: khi `6` đến chỉ giải quyết `4`, bỏ sót `5`. Sửa: dùng `while`.

### Lỗi 2: xem `stack[-1]` khi stack rỗng
`while stack[-1] < x and stack:` → `IndexError` ở lần đầu. Phải viết `while stack and stack[-1] < x:` (kiểm tra rỗng **trước**).

### Lỗi 3: lấy số lớn nhất bên phải thay vì số đầu tiên
`max(nums2[j+1:])` sai với `[1, 3, 5]` (trả 5, đúng phải là 3).

### Lỗi 4: tìm next greater trong `nums1` thay vì `nums2`
`nums1` chỉ là danh sách câu hỏi. Thứ tự "bên phải" được định nghĩa trong `nums2`.

In [19]:
def buggy_if(nums1, nums2):
    nxt, stack = {}, []
    for x in nums2:
        if stack and stack[-1] < x:
            nxt[stack.pop()] = x
        stack.append(x)
    return [nxt.get(x, -1) for x in nums1]

def buggy_max(nums1, nums2):
    ans = []
    for x in nums1:
        right = [y for y in nums2[nums2.index(x) + 1:] if y > x]
        ans.append(max(right) if right else -1)
    return ans

print("if thay while:", buggy_if([5, 4], [5, 4, 6]), "| đúng:", next_greater_stack([5, 4], [5, 4, 6]))
print("lấy max      :", buggy_max([1], [1, 3, 5]), "| đúng:", next_greater_stack([1], [1, 3, 5]))
try:
    stack = []
    while stack[-1] < 3 and stack:
        pass
except IndexError as e:
    print("Sai thứ tự điều kiện -> IndexError:", e)

if thay while: [-1, 6] | đúng: [6, 6]
lấy max      : [5] | đúng: [3]
Sai thứ tự điều kiện -> IndexError: list index out of range


## 14. Mẹo nhận dạng pattern Monotonic Stack

Nghĩ tới monotonic stack khi đề hỏi về **"phần tử gần nhất bên trái/phải thoả lớn hơn/nhỏ hơn"**:

| Đề hỏi | Stack giữ | Pop khi |
|---|---|---|
| next greater (bên phải, lớn hơn) | giảm dần | `stack[-1] < x` |
| next smaller (bên phải, nhỏ hơn) | tăng dần | `stack[-1] > x` |
| khoảng cách tới ngày ấm hơn (Daily Temperatures) | **chỉ số**, giá trị giảm dần | `T[stack[-1]] < T[i]` |
| mảng vòng tròn (NGE II) | duyệt `2n` lần với `i % n` | như trên |

Dấu hiệu: brute force là hai vòng lặp "với mỗi i, quét j sang phải tới khi gặp ...". Khi đó thử: "để phần tử mới giải quyết các phần tử đang chờ".

**Mẹo lưu chỉ số thay vì giá trị:** khi mảng có thể trùng giá trị hoặc cần khoảng cách, đẩy **index** vào stack.

## 15. Biến thể

### 15.1. Next Greater Element II (LeetCode 503) — mảng vòng tròn
Sau phần tử cuối thì quay lại đầu. Mảng có thể **trùng giá trị**, nên ta trả về mảng theo **chỉ số** và lưu chỉ số trong stack. Trick: duyệt `2n` lần, dùng `i % n`.

In [20]:
def next_greater_circular(nums):
    n = len(nums)
    ans = [-1] * n
    stack = []                       # lưu CHỈ SỐ
    for i in range(2 * n):
        x = nums[i % n]
        while stack and nums[stack[-1]] < x:
            ans[stack.pop()] = x
        if i < n:
            stack.append(i)
    return ans

print(next_greater_circular([1, 2, 1]))        # [2, -1, 2]
print(next_greater_circular([1, 2, 3, 4, 3]))  # [2, 3, 4, -1, 4]

[2, -1, 2]
[2, 3, 4, -1, 4]


### 15.2. Daily Temperatures (LeetCode 739) — đếm số ngày phải chờ
Thay vì giá trị, ta cần **khoảng cách** `i - j` → stack lưu chỉ số.

In [21]:
def daily_temperatures(temps):
    ans = [0] * len(temps)
    stack = []
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()
            ans[j] = i - j
        stack.append(i)
    return ans

print(daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]))  # [1, 1, 4, 2, 1, 1, 0, 0]

[1, 1, 4, 2, 1, 1, 0, 0]


## 16. Bài tập (tự làm trước, đáp án ở cuối)

### Bài 1 — Tính tay
Với `nums2 = [6, 2, 9, 1, 3, 8]`, hãy lập bảng trace stack trái→phải như mục 7 và ghi next greater của từng số. (Không cần code; tự kiểm tra bằng `trace_next_greater` sau khi làm xong.)

### Bài 2 — Điền TODO
Hoàn thiện lời giải monotonic stack.

In [22]:
def exercise_next_greater(nums1, nums2):
    next_greater = {}
    stack = []
    for x in nums2:
        # TODO 1: trong khi stack còn và đỉnh nhỏ hơn x -> pop và ghi next_greater
        # TODO 2: đẩy x vào stack
        pass
    # TODO 3: trả về list kết quả, mặc định -1
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_next_greater([4, 1, 2], [1, 3, 4, 2]) == [-1, 3, -1]

### Bài 3 — Next Smaller Element
Với mỗi phần tử của `nums` (giá trị unique), tìm phần tử **đầu tiên bên phải nhỏ hơn** nó, không có thì `-1`. Trả về list cùng độ dài.

In [23]:
def next_smaller(nums):
    # TODO: stack tăng dần
    pass

# assert next_smaller([4, 8, 5, 2, 25]) == [2, 5, 2, -1, -1]

### Bài 4 — Previous Greater Element
Với mỗi phần tử, tìm phần tử **gần nhất bên trái lớn hơn** nó, không có thì `-1`.

In [24]:
def previous_greater(nums):
    # TODO: duyệt trái -> phải, stack giữ ứng viên ở bên trái
    pass

# assert previous_greater([2, 7, 3, 5, 4, 6, 8]) == [-1, -1, 7, 7, 5, 7, -1]

### Bài 5 — Next Greater có trùng giá trị
Cho `nums` **có thể trùng**, trả về list `ans` với `ans[i]` là next greater (nghiêm ngặt lớn hơn) của `nums[i]`. Gợi ý: dict theo giá trị không còn dùng được.

In [25]:
def next_greater_with_duplicates(nums):
    # TODO
    pass

# assert next_greater_with_duplicates([2, 2, 1, 3, 3]) == [3, 3, 3, -1, -1]

### Bài 6 — Mini challenge: Online Stock Span (gọn)
Cho list giá `prices`, với mỗi ngày `i` tính **span** = số ngày liên tiếp kết thúc ở `i` (tính cả `i`) có giá `<= prices[i]`.

In [26]:
def stock_spans(prices):
    # TODO: stack lưu chỉ số các ngày có giá lớn hơn
    pass

# assert stock_spans([100, 80, 60, 70, 60, 75, 85]) == [1, 1, 1, 2, 1, 4, 6]

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Ai còn chờ khi `9` đến? | Stack giảm dần | `9` pop `2` và `6` |
| 2 | Xem mục 6 | `while stack and stack[-1] < x` | `return [next_greater.get(x, -1) for x in nums1]` |
| 3 | Đổi chiều so sánh | stack **tăng** dần | `while stack and nums[stack[-1]] > x: ans[stack.pop()] = x` |
| 4 | Đáp án nằm bên trái → xét lúc **đẩy vào** | pop mọi số `<= x` | sau khi pop: `ans[i] = stack[-1] if stack else -1` |
| 5 | Trùng giá trị → dùng chỉ số | `ans = [-1]*n`, stack lưu `i` | `while stack and nums[stack[-1]] < x: ans[stack.pop()] = x` |
| 6 | Span = khoảng cách tới previous greater | stack lưu chỉ số giá **lớn hơn** | `span = i - stack[-1] if stack else i + 1` |

## 18. Đáp án tham khảo

**Bài 1:** `nums2 = [6, 2, 9, 1, 3, 8]` → `6→9, 2→9, 9→-1, 1→3, 3→8, 8→-1`. Kiểm tra bằng trace:

In [27]:
trace_next_greater([6, 2, 9, 1, 3, 8], [6, 2, 9, 1, 3, 8])

bước 1: x= 6 | stack trước=[]           | gán: —          | stack sau=[6]
bước 2: x= 2 | stack trước=[6]          | gán: —          | stack sau=[6, 2]
bước 3: x= 9 | stack trước=[6, 2]       | gán: 2->9, 6->9 | stack sau=[9]
bước 4: x= 1 | stack trước=[9]          | gán: —          | stack sau=[9, 1]
bước 5: x= 3 | stack trước=[9, 1]       | gán: 1->3       | stack sau=[9, 3]
bước 6: x= 8 | stack trước=[9, 3]       | gán: 3->8       | stack sau=[9, 8]
Còn chờ (không có next greater): [9, 8]
next_greater = {2: 9, 6: 9, 1: 3, 3: 8}
ans = [9, 9, -1, 3, 8, -1]


[9, 9, -1, 3, 8, -1]

In [28]:
def answer_exercise_next_greater(nums1, nums2):
    next_greater = {}
    stack = []
    for x in nums2:
        while stack and stack[-1] < x:
            next_greater[stack.pop()] = x
        stack.append(x)
    return [next_greater.get(x, -1) for x in nums1]

def answer_next_smaller(nums):
    ans = [-1] * len(nums)
    stack = []
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] > x:
            ans[stack.pop()] = x
        stack.append(i)
    return ans

def answer_previous_greater(nums):
    ans = []
    stack = []
    for x in nums:
        while stack and stack[-1] <= x:
            stack.pop()
        ans.append(stack[-1] if stack else -1)
        stack.append(x)
    return ans

def answer_next_greater_with_duplicates(nums):
    ans = [-1] * len(nums)
    stack = []
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            ans[stack.pop()] = x
        stack.append(i)
    return ans

def answer_stock_spans(prices):
    spans = []
    stack = []   # chỉ số các ngày có giá > giá hiện tại
    for i, p in enumerate(prices):
        while stack and prices[stack[-1]] <= p:
            stack.pop()
        spans.append(i - stack[-1] if stack else i + 1)
        stack.append(i)
    return spans

assert answer_exercise_next_greater([4, 1, 2], [1, 3, 4, 2]) == [-1, 3, -1]
assert answer_next_smaller([4, 8, 5, 2, 25]) == [2, 5, 2, -1, -1]
assert answer_previous_greater([2, 7, 3, 5, 4, 6, 8]) == [-1, -1, 7, 7, 5, 7, -1]
assert answer_next_greater_with_duplicates([2, 2, 1, 3, 3]) == [3, 3, 3, -1, -1]
assert answer_stock_spans([100, 80, 60, 70, 60, 75, 85]) == [1, 1, 1, 2, 1, 4, 6]
print("Đáp án bài 2–6: PASS toàn bộ test!")

Đáp án bài 2–6: PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
# Next greater (bên phải, lớn hơn) — stack giảm dần
stack = []
for x in arr:
    while stack and stack[-1] < x:   # x giải quyết các số đang chờ
        result[stack.pop()] = x
    stack.append(x)                  # x bắt đầu chờ
# phần tử còn lại trong stack -> -1
```

Câu thần chú: **"Số mới đến, pop hết những ai nhỏ hơn nó — nó chính là câu trả lời của họ."**

### Checklist trước khi submit
- [ ] Tìm next greater trong `nums2`, trả lời theo thứ tự `nums1`.
- [ ] `while` (không phải `if`) và kiểm tra `stack` **trước** `stack[-1]`.
- [ ] Phần tử không có đáp án trả `-1` (dùng `.get(x, -1)`).
- [ ] So sánh nghiêm ngặt `<` (lớn hơn, không phải lớn hơn hoặc bằng).
- [ ] Nếu biến thể có trùng giá trị/cần khoảng cách → lưu **chỉ số** trong stack.
- [ ] Time `O(n + m)`, Space `O(n)`.